# BM25와 Dense Retrieval 비교

BM25와 Dense Retrieval은 관련 문서를 찾는 기준이 다르다.

- `BM25(Best Matching 25)`: 단어 빈도·희귀도·문서 길이를 이용하는 키워드 기반 검색 방법이다.
- 희소 검색(Sparse Retrieval): 문서에 실제로 등장한 단어를 중심으로 비교하는 검색이다.
- Dense Retrieval: 질의와 문서를 임베딩 벡터로 바꿔 의미를 비교하는 검색이다.
- 임베딩 벡터: 텍스트의 의미를 숫자 배열로 표현한 값이다.
- cosine 유사도: 두 벡터의 방향이 얼마나 비슷한지 측정하며 값이 클수록 의미가 가깝다.
- Vector DB: 임베딩 벡터와 원문·문서 식별 정보를 저장하고 유사한 벡터를 검색하는 데이터베이스이다.
- Pinecone: Dense Retrieval에서 사용할 클라우드 Vector DB 서비스이다.
- Vector Store: 코드에서 벡터 저장·검색을 다루는 LangChain 연결 객체이다.

BM25는 고유명사·정확한 키워드에, Dense Retrieval은 단어가 달라도 뜻이 가까운 문서에 강점이 있다. BM25 점수와 cosine 유사도는 기준이 달라 숫자를 직접 비교하지 않는다.

데이터 흐름은 `문서·질의 → BM25 또는 Dense Retrieval → 문서 ID 순위 → 공통 평가지표`이다.

## 평가 지표

- Precision@5(P@5): 상위 5개 결과 중 관련 문서의 비율이다.
- Recall@5(R@5): 전체 관련 문서 중 상위 5개에서 찾은 비율이다.
- MRR(Mean Reciprocal Rank): 첫 관련 문서 순위의 역수를 질의 전체에서 평균한 값이다.
- MAP@5(Mean Average Precision at 5): 관련 문서가 상위 5개의 앞쪽에 배치된 정도를 질의 전체에서 평균한 값이다.

## BM25·Dense 비교 실행 패키지 준비

형태소 기반 BM25와 OpenAI·Pinecone 기반 Dense 검색을 실행하기 위해 관련 패키지를 준비한다.

- `pandas`: CSV(쉼표로 열을 나눈 표 파일)를 DataFrame(행·열 표 객체)으로 읽고 다룬다.
- `numpy`: 질의별 평가 지표를 배열로 만들고 평균한다.
- `rank_bm25`: BM25 점수와 순위를 계산한다.
- `KoNLPy`: 한국어 형태소 분석기 `Okt(Open Korean Text)`를 제공한다.
- `langchain-openai`: OpenAI 임베딩을 LangChain에서 사용한다.
- `langchain-pinecone`: Pinecone을 LangChain Vector Store로 연결한다.
- `pinecone`: Pinecone 서비스와 통신하는 Python 패키지이다.
- `python-dotenv`: key·모델 설정을 코드 밖에 저장한 `.env` 파일을 읽는다.
- `gdown`: Google Drive 파일을 ID로 내려받는다.


In [1]:
%pip install -U pandas numpy rank_bm25 konlpy langchain-openai langchain-pinecone pinecone python-dotenv gdown

  Using cached konlpy-0.6.0-py2.py3-none-any.whl.metadata (1.9 kB)
  Using cached pinecone-9.1.0-cp310-abi3-win_amd64.whl.metadata (6.3 kB)
  Using cached jpype1-1.7.1-cp312-cp312-win_amd64.whl.metadata (4.3 kB)
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
   -------------- ------------------------- 4.5/12.5 MB 22.3 MB/s eta 0:00:01
   ------------------------------- -------- 10.0/12.5 MB 23.9 MB/s eta 0:00:01
   ---------------------------------------- 12.5/12.5 MB 20.0 MB/s  0:00:00
Using cached konlpy-0.6.0-py2.py3-none-any.whl (19.4 MB)
Using cached jpype1-1.7.1-cp312-cp312-win_amd64.whl (356 kB)

  Attempting uninstall: numpy

    Found existing installation: numpy 2.5.1

   ---------------------------------------- 0/4 [numpy]
   ---------------------------------------- 0/4 [numpy]
   ---------------------------------------- 0/4 [numpy]
   ---------------------------------------- 0/4 [numpy]
    Uninstalling numpy-2.5.1:
   --------------------------------

## 문서·질의·정답 데이터 다운로드

두 검색기에 같은 문서·질의·정답을 제공해 공정한 비교를 준비한다.

- CSV(Comma-Separated Values): 열을 쉼표로 구분한 표 형식의 텍스트 파일이다.
- `gdown`: Google Drive 파일 ID로 파일을 내려받는 명령이다.
- `--output`: 내려받은 파일의 저장 이름을 지정한다.


In [2]:
# 검색 대상 문서
!gdown 1pspaw_q4_QCp2K4M-thDlUtra-_wQI7k --output documents.csv

# 질의와 관련 문서 정답
!gdown 1dsn0pwkfzOUxiQ4MKDIvMM-CkxbYiSce --output queries.csv

Downloading...
From: https://drive.google.com/uc?id=1pspaw_q4_QCp2K4M-thDlUtra-_wQI7k
To: D:\SKN_AI\09_llm\07_advancewd_rag\01_retrieval_optimization\documents.csv

  0%|          | 0.00/14.9k [00:00<?, ?B/s]
100%|██████████| 14.9k/14.9k [00:00<?, ?B/s]
Downloading...
From: https://drive.google.com/uc?id=1dsn0pwkfzOUxiQ4MKDIvMM-CkxbYiSce
To: D:\SKN_AI\09_llm\07_advancewd_rag\01_retrieval_optimization\queries.csv

  0%|          | 0.00/2.19k [00:00<?, ?B/s]
100%|██████████| 2.19k/2.19k [00:00<00:00, 1.86MB/s]


## 검색 문서와 질의 데이터 읽기

`documents.csv`는 검색 문서를, `queries.csv`는 검색 질의와 평가 기준을 담고 있다.

- DataFrame: 행과 열로 구성된 Pandas 표 객체이다.

- `doc_id`: 문서 식별자이다.
- `content`: 검색 대상 문서 본문이다.
- `query_id`: 질의 식별자이다.
- `query_text`: 검색에 사용할 질의 문장이다.




In [1]:
import pandas as pd

documents_df = pd.read_csv('documents.csv')
queries_df = pd.read_csv('queries.csv')

queries_df.head(3)

,query_id,query_text,relevant_doc_ids
0,Q1,제주도 올레길 트레킹 코스 추천,D1=3;D4=1;D30=1
1,Q2,전주 비빔밥 vs 진주 비빔밥 재료 및 맛 차이,D2=3
2,Q3,걸스데이 대표 히트곡 목록 알려줘,D3=3


## 형태소 기반 BM25 인덱스와 검색 함수 구성

문서와 질의를 같은 형태소 기준으로 나눈 뒤 BM25 점수를 계산한다.

- `Okt(Open Korean Text)`: 한국어 형태소 분석기이다.
- `Okt.morphs()`: 한국어 문자열을 형태소 목록으로 변환한다.
- `BM25Okapi`: 토큰화한 문서를 색인하고 BM25 점수를 계산한다.
- IDF(Inverse Document Frequency): 여러 문서에서 드물게 등장한 단어에 더 큰 값을 주는 지표이다.
- BM25 score: 값이 클수록 질의와 관련성이 높다.

### 상위 문서

상위 문서는 BM25 점수가 높은 순서로 선택된 검색 후보이다.

- 선택 대상: BM25 점수가 0보다 큰 문서이다.
- 정렬 기준: BM25 점수 내림차순이다.
- 선택 개수: 최대 `top_k`개이다.
- 주의: 검색 후보이며 실제 정답 문서를 보장하지 않는다.

검색 순서:

1. 질의를 형태소로 나눈다.
2. 문서별 BM25 점수를 계산한다.
3. 양수 점수 후보를 내림차순으로 정렬한다.
4. 상위 `top_k`개의 `doc_id`를 반환한다.


In [6]:
from konlpy.tag import Okt
from rank_bm25 import BM25Okapi

okt = Okt() # 한국어 형태소 분석기 생성

# document['content']를 형태소 목록으로 분해
tokenized_documents = [
    okt.morphs(content)
    for content in documents_df['content']
]

# print(tokenized_documents[0])

# 토근화된 문서를 BM250kapi를 이용해서 색인화
# -> 형태소 검색을 할 수 있는 객체 + bm25 점수 측정
bm25 = BM25Okapi(
    # 문서별 형태소 목록
    corpus=tokenized_documents,

    # k1: 같은 단어의 반복 출현을 점수에 얼마나 반영할지 설정
    k1=1.5,

    # b: 긴 문서가 지나치게 유리해지는 것을 보정하는 강도
    b=0.75,

    # 너무 흔해서 음수가된 IDF를 보정하는 비율
    epsilon=0.25
)

# BM25 점수 문서를 내림 차순으로 최대 top-k개 만큼 반환
def bm25_search(query:str, top_k:int = 5) -> list[dict]:

    # 1. 전달 받은 질문(query)를 형태소 목록으로 변환
    query_tokens = okt.morphs(query)
    # print(query_tokens)

    # 2. 형태소 목록을 문서별 bm25 점수 배열로 변환
    document_scores = bm25.get_scores(query_tokens)
    # print(document_scores)

    # 3. document_scores가 0 초과하는 인덱스만 True로 표시
    # nonzero(): True인 위치의 축별 인덱스를 반환
    positive_indices = (document_scores > 0).nonzero()[0].tolist()
    # print(positive_indices)

    # 4. ranked_indices: 점수 내림차순으로 정렬한 상위 k개 행 번호
    ranked_indices = sorted(
        positive_indices,
        reverse=True,
        key=lambda index: document_scores[index]
    )[:top_k]

    return [
        {
            'rank': rank,
            'doc_id': documents_df.iloc[index]['doc_id'],
            'score': document_scores[index],
        }
        for rank, index in enumerate(ranked_indices, start=1)
    ]

bm25_sample = bm25_search("제주도 관광 명소", top_k=5)
pd.DataFrame(bm25_sample)

# BM25 높은 점수를 받는 조건
#1) query의 형태소와 등록된 Document의 형태소가 일치하는게 많다
#2) 일치한 형태소가 전체 문서에서 드문 단어이다.
#3) 해당 형태소가 문서에 적절히 반복된다.
# 4) 문서 길이를 고려해도 단어 비중이 높다.

# BM25 저 ㅁ수 범위에 따른 해석 방법
# -0.5(음수) 문서 전체에서 너무 흔한 단어
# -0.0 : 유효한 일치가 거의 없음
# -1.2(작은 양수):어느 정도 관련된 후보
# -5.6 (큰 양수) ㅣ 관련성이 높은 후보

,rank,doc_id,score
0,1,D1,5.631702
1,2,D12,2.546678


## BM25 2순위 후보 D12의 본문 확인

BM25 검색 결과 상위 후보 중 2순위인 D12의 본문을 확인한다.

D12는 `여행`, `관광`과 같은 단어 때문에 검색됐지만, 제주도가 아닌 서울 근교 여행을 설명한다.

이를 통해 BM25 점수가 높더라도 질문의 의도와 정확히 일치하지 않을 수 있음을 확인한다.

예를 들어 다른 지역의 여행 문서는 `여행`, `관광`이라는 단어 때문에 높은 점수를 받을 수 있지만, 제주도 질문과 관련 없는 문서이다.

In [7]:
d12_content = documents_df.loc[documents_df['doc_id']=='D12','content']

d12_content

11    서울 근교에서 당일치기로 다녀올 만한 여행지로는 가평 쁘띠프랑스, 남양주 수종사, ...
Name: content, dtype: str

## 모든 질의의 BM25 검색 결과 저장

각 질의를 BM25로 검색하고, 점수가 높은 문서를 최대 5개까지 저장한다.

- `bm25_rankings`: 순위, 문서 ID, BM25 점수를 저장한다.
- `bm25_results`: 검색 성능 평가에 사용할 문서 ID 순서만 저장한다.
- BM25 점수가 0보다 큰 문서가 2개라면 2개만 저장한다.
- 결과 수를 5개로 맞추기 위해 관련 없는 0점 문서를 추가하지 않는다.

In [11]:
# queries_df에 존재하는 30개 질의에 대한
# bm25_search() 결과를 반환 받아서 ID목록 만들기

bm25_rankings={}
bm25_results={}

for _, row in queries_df.iterrows():
    query_id = row['query_id']
    query_text = row['query_text']

    ranked_items = bm25_search(query_text, top_k=5)

    bm25_rankings[query_id]=ranked_items

    bm25_results[query_id]=[item['doc_id'] for item in ranked_items ] # 평가 지표 함수에 전달할 doc_id 목록

display(pd.DataFrame(bm25_rankings['Q2']))

,rank,doc_id,score
0,1,D13,19.251204
1,2,D2,18.003683
2,3,D10,2.503598
3,4,D27,2.448375
4,5,D9,2.062554


## Dense Retrieval

Dense Retrieval은 질의와 문서를 임베딩 벡터로 변환하고, 의미적으로 가까운 문서를 찾는 검색 방식이다.

- 임베딩 벡터: 텍스트의 의미를 숫자 배열로 표현한 값이다.
- Dense: 벡터의 많은 차원에 값이 들어 있는 밀집 형태를 뜻한다.
- 검색 기준: 질의 벡터와 문서 벡터의 cosine 유사도이다.
- 장점: 사용된 단어가 달라도 의미가 비슷한 문서를 찾을 수 있다.
- BM25와의 차이: BM25는 단어 일치, Dense Retrieval은 의미 유사성을 중심으로 검색한다.

검색 흐름:

1. 문서는 `01_indexing.ipynb`에서 임베딩해 Pinecone에 저장한다.
2. 검색할 질의도 같은 임베딩 모델로 벡터화한다.
3. Pinecone이 질의 벡터와 문서 벡터의 유사도를 계산한다.
4. 유사도 점수가 높은 문서를 검색 결과로 반환한다.


## Dense Retrieval 환경 설정

Dense Retrieval은 문서를 저장할 때와 같은 설정으로 Pinecone에 연결해야 한다.

- API(Application Programming Interface) key: OpenAI·Pinecone 요청 주체를 인증하는 비밀 값이다.
- `.env`: API key와 모델·index 설정을 코드 밖에 저장하는 파일이다.
- index: Pinecone에서 벡터를 저장하는 최상위 검색 단위이다.
- namespace: 하나의 index 안에서 벡터를 나누는 논리적 검색 영역이다.
- 임베딩 모델: 문서와 질의에 모두 같은 모델을 사용한다.
- 벡터 차원: Pinecone index와 임베딩 출력의 길이가 같아야 한다.

이 단원은 `01_indexing.ipynb`에서 사용한 값을 그대로 불러와 기존 index를 Vector Store에 연결한다.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=False)

PINECONE_INDEX_NAME = os.getenv('PINECONE_INDEX_NAME', 'adv-rag')
PINECONE_NAMESPACE = os.getenv('PINECONE_NAMESPACE', '')
OPENAI_EMBEDDING_MODEL = os.getenv(
    'OPENAI_EMBEDDING_MODEL',
    'text-embedding-3-small',
).strip() or 'text-embedding-3-small'
PINECONE_INDEX_DIMENSION = int(os.getenv('PINECONE_INDEX_DIMENSION', '1536'))

## OpenAI 임베딩과 Pinecone Vector Store 연결

`OpenAIEmbeddings`는 질의를 문서와 같은 벡터 공간으로 변환한다. `PineconeVectorStore`는 지정한 index와 namespace에서 가까운 문서를 검색한다.

- `Document`: LangChain의 문서 객체이다.
- `page_content`: 검색된 문서 본문이다.
- `metadata`: `doc_id`처럼 검색 결과를 원문과 연결하는 부가 정보 딕셔너리이다.

이 단계에서는 문서를 다시 저장하지 않고 기존 Pinecone index를 검색용 Vector Store로 연결한다.


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

## 모든 질의의 Dense 검색 결과 저장

30개 질의를 Dense Retrieval로 검색하고 결과를 두 가지 형태로 저장한다.

검색 순서:

1. `query_text`를 임베딩 벡터로 변환한다.
2. Pinecone의 문서 벡터와 cosine 유사도를 계산한다.
3. 유사도 점수가 높은 문서를 최대 5개까지 반환한다.

저장 결과:

- `dense_rankings`: 순위·문서 ID·cosine score를 저장한다.
- `dense_results`: 평가에 사용할 문서 ID 순서만 저장한다.
- cosine score: 값이 클수록 의미가 가깝지만 확률값으로 해석하지 않는다.


## qrels

`qrels`는 질의별 관련 문서를 기록한 **검색 평가용 정답표**이다.

- 정식 이름: Query Relevance Judgments이다.
- 읽는 법: 보통 "큐렐즈"라고 읽는다.
- 현재 열: `relevant_doc_ids`이다.
- 저장 형식: `문서 ID=관련성 등급`이다.
- 예시: `D1=3;D4=1;D30=1`은 세 문서가 관련 문서라는 뜻이다.
- 관련성 등급: 숫자가 클수록 질의와 더 관련 있다는 뜻이다.
- 사용 위치: 검색 결과와 비교해 Precision·Recall·MRR·MAP을 계산한다.

> qrels는 검색기가 만든 결과가 아니라, 검색 성능을 평가하기 위해 미리 준비한 정답이다.

## qrels 문자열을 관련 문서 딕셔너리로 변환

앞에서 확인한 `relevant_doc_ids` 문자열을 평가 함수가 사용할 딕셔너리로 변환한다.

- 입력: `D1=3;D4=1;D30=1`이다.
- 분리: `;`로 문서를 나누고 `=`로 문서 ID와 등급을 나눈다.
- 출력: `{'D1': 3, 'D4': 1, 'D30': 1}`이다.
- 평가 기준: 등급이 1 이상인 문서를 관련 문서로 사용한다.
- binary relevance: 등급을 관련·비관련 두 범주로 바꿔 평가하는 방식이다.


## 모든 질의의 qrels 변환 확인

각 `query_id`에 관련 문서 딕셔너리를 연결한다.

- hit: 검색된 `doc_id`가 해당 질의의 qrels에 포함된 경우이다.
- 평가 연결: 같은 `query_id`로 검색 결과와 qrels를 찾아 hit 여부를 계산한다.


## 한 질의의 검색 순위를 평가하는 네 지표

`compute_metrics()`는 문서 ID 순위와 관련 문서 딕셔너리로 한 질의의 검색 품질을 계산한다. 결과가 5개보다 짧아도 Precision@5의 분모는 5이므로 찾지 못한 문서가 반영된다.

- Precision@k: 상위 k개 중 관련 문서의 비율이다.
- Recall@k: 전체 관련 문서 중 상위 k개에서 찾은 비율이다.
- RR(Reciprocal Rank): 첫 관련 문서 순위의 역수이다.
- AP@k(Average Precision at k): 관련 문서가 상위 k개의 앞쪽에 배치된 정도를 반영한다.


## Q1의 BM25 결과로 지표 계산 확인

Q1의 BM25 문서 ID 순위와 qrels를 먼저 입력해 네 반환값의 순서와 범위를 확인한다. 같은 계산을 다음 셀에서 모든 질의에 반복한다.


## 전체 질의의 평균 검색 품질 계산

`evaluate_all()`은 질의별 지표를 계산한 뒤 같은 지표끼리 평균한다.

- MRR(Mean Reciprocal Rank): 질의별 RR의 평균이다.
- MAP@k(Mean Average Precision at k): 질의별 AP@k의 평균이다.


In [ ]:
import numpy as np

## BM25와 Dense Retrieval의 지표 비교

두 검색 결과를 같은 `queries_df`, 같은 `k=5`로 평가한다. BM25 score와 cosine score는 직접 비교하지 않고, 정확성·회수율·첫 정답 순위·여러 정답의 배치를 보여 주는 공통 지표를 비교한다.
